# Cross-Validation & Hyperparameter Tuning

## The Problem
- One train-test split might be lucky or unlucky
- How do we know if our model is truly good?

## Cross-Validation Solution
Split data into K parts. Train on K-1, test on 1. Repeat K times. Average results.

## Hyperparameters
Settings we choose BEFORE training (not learned from data):
- K in KNN
- max_depth in Decision Tree
- n_estimators in Random Forest

## GridSearchCV
Try ALL combinations of hyperparameters and pick the best!

In [1]:
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
import numpy as np

# Load data
iris = load_iris()
X, y = iris.data, iris.target

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
# Cross-Validation: 5-Fold
rf = RandomForestClassifier(random_state=42)

# Perform 5-fold cross-validation
cv_scores = cross_val_score(rf, X_train, y_train, cv=5)

print("5-Fold CV Scores:", cv_scores)
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Std Dev: {cv_scores.std():.4f}")
print(f"95% Confidence Interval: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

5-Fold CV Scores: [0.95833333 0.95833333 0.83333333 1.         0.95833333]
Mean CV Accuracy: 0.9417
Std Dev: 0.0565
95% Confidence Interval: 0.9417 (+/- 0.1130)


In [3]:
# GridSearchCV: Find best hyperparameters

param_grid = {
    'n_estimators': [10, 50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1  # Use all CPU cores
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Test on held-out test set
best_model = grid_search.best_estimator_
test_score = best_model.score(X_test, y_test)
print(f"Test Score with Best Model: {test_score:.4f}")

Best Parameters: {'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 50}
Best CV Score: 0.9500
Test Score with Best Model: 1.0000


In [4]:
# Compare: Default vs Tuned Model

# Default
default_rf = RandomForestClassifier(random_state=42)
default_rf.fit(X_train, y_train)
default_score = default_rf.score(X_test, y_test)

# Tuned
tuned_score = test_score

print(f"Default Model Accuracy: {default_score:.4f}")
print(f"Tuned Model Accuracy: {tuned_score:.4f}")
print(f"Improvement: {tuned_score - default_score:.4f}")

Default Model Accuracy: 1.0000
Tuned Model Accuracy: 1.0000
Improvement: 0.0000
